# unify_pums.ipynb

This notebook reads in the household and person-level PUMS for a given year, merges them, cleans up the ORIGIN/CHOSEN fields, and injects fields that will be used later on during modeling.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("../.."))
from lib import io as lio


In [2]:
year = 2018
directory = "us"
base_household_name = "psam_hus"
base_person_name = "psam_pus"

In [3]:
person_cols = set(
    pd.read_csv(f"{directory}/{base_person_name}a_{year}.csv", nrows=0).columns
)
household_cols = set(
    pd.read_csv(f"{directory}/{base_household_name}a_{year}.csv", nrows=0).columns
)

In [4]:
for col in household_cols:
    print(col)

BROADBND
FFINCP
FCOMPOTHXP
VALP
AGS
WGTP4
FSMOCP
FINCP
WGTP23
WGTP49
WGTP56
YBL
WGTP80
WGTP21
FBDSP
FGRNTP
FYBLP
WGTP64
WGTP47
WGTP13
WGTP53
FRWATP
FKITP
STOV
ELEP
FINSP
MHP
FULP
MRGI
TYPE
FSMXSP
WGTP38
HUGCL
WGTP65
WGTP15
WKEXREL
FTABLETP
WGTP73
WGTP51
FHINCP
FCONP
RWAT
FFSP
WGTP31
ELEFP
PLMPRP
WGTP41
FACCESSP
FSMP
WGTP61
RNTP
FACRP
WGTP48
FBROADBNDP
CONP
RT
WGTP39
RWATPR
REGION
WGTP60
SMARTPHONE
BATH
WGTP75
WGTP24
WGTP74
WGTP1
FGASP
HFL
FSMXHP
NOC
TEN
WGTP10
WGTP35
WGTP77
WGTP79
WATP
FELEP
NP
SERIALNO
FPARC
WGTP7
NPF
FVEHP
FAGSP
FWATP
R65
TEL
FRMSP
WGTP28
FMRGXP
WORKSTAT
FVACSP
WGTP29
WGTP44
FVALP
LNGI
WGTP26
RESMODE
BDSP
REFR
DIVISION
WGTP36
FMRGIP
WGTP67
WATFP
OCPIP
SMX
NRC
HUPAC
R60
WGTP12
WGTP45
FMRGP
FTAXP
PLM
WGTP22
MRGX
GRNTP
MULTG
R18
WGTP42
WGTP57
WGTP34
DIALUP
SVAL
WGTP20
WGTP5
FSINKP
FLAPTOPP
HUPAOC
FS
WGTP16
FSATELLITEP
WGTP9
ACCESS
FRNTP
WIF
HHLANP
WGTP63
WGTP59
WGTP50
INSP
FDIALUPP
ADJINC
FSTOVP
WGTP
ST
WGTP40
MV
FREFRP
WGTP78
WGTP68
ACR
VEH
WGTP54
MRGT
SINK
FHFLP
RNTM


In [5]:
def filter_person_cols(col: str):
    # weight column
    if col.startswith("PWGTP") and col != "PWGTP":
        return False
    # filter out flag columns
    return col[1:-1] not in person_cols


target_household_cols = {
    "SERIALNO",
    "NP",
    "TYPE",
    "TEN",
    "VALP",
    "VEH",
    "FES",
    "FINCP",
    "FPARC",
    "GRNTP",
    "GRPIP",
    "HHT",
    "HINCP",
    "OCPIP",
    "PARTNER",
    "R18",
    "SMOCP",
    "TAXAMT",
    "WIF",
    "HUPAOC",
    "HUPARC",
    "MULTG",
    "MV",
    "R65",
    "ACR",
    "MRGP",
    "MRGT",
    "NOC",
    "WKEXREL",
    "WORKSTAT",
    "HUGCL",
    "NPF",
    "NPP",
    "NR",
    "NRC",
    # "CPLT",
}


def filter_household_cols(col: str):
    # weight column
    if col.startswith("WGTP"):
        return False
    # flag column
    if col[1:-1] in person_cols:
        return False
    return col in target_household_cols

In [6]:
# reading in the individual PUMS
dfs = []
dfs.append(
    pd.read_csv(
        f"{directory}/{base_person_name}a_{year}.csv", usecols=filter_person_cols
    )
)
dfs.append(
    pd.read_csv(
        f"{directory}/{base_person_name}b_{year}.csv", usecols=filter_person_cols
    )
)
df_p = pd.concat(dfs).reset_index(drop=True)

del dfs

In [7]:
for col in df_p.columns:
    print(col)

RT
SERIALNO
DIVISION
SPORDER
PUMA
REGION
ST
ADJINC
PWGTP
AGEP
CIT
CITWP
COW
DDRS
DEAR
DEYE
DOUT
DPHY
DRAT
DRATX
DREM
ENG
FER
GCL
GCM
GCR
HINS1
HINS2
HINS3
HINS4
HINS5
HINS6
HINS7
INTP
JWMNP
JWRIP
JWTR
LANX
MAR
MARHD
MARHM
MARHT
MARHW
MARHYP
MIG
MIL
MLPA
MLPB
MLPCD
MLPE
MLPFG
MLPH
MLPI
MLPJ
MLPK
NWAB
NWAV
NWLA
NWLK
NWRE
OIP
PAP
RELP
RETP
SCH
SCHG
SCHL
SEMP
SEX
SSIP
SSP
WAGP
WKHP
WKL
WKW
WRK
YOEP
ANC
ANC1P
ANC2P
DECADE
DIS
DRIVESP
ESP
ESR
FOD1P
FOD2P
HICOV
HISP
INDP
JWAP
JWDP
LANP
MIGPUMA
MIGSP
MSP
NAICSP
NATIVITY
NOP
OC
OCCP
PAOC
PERNP
PINCP
POBP
POVPIP
POWPUMA
POWSP
PRIVCOV
PUBCOV
QTRBIR
RAC1P
RAC2P
RAC3P
RACAIAN
RACASN
RACBLK
RACNH
RACNUM
RACPI
RACSOR
RACWHT
RC
SCIENGP
SCIENGRLP
SFN
SFR
VPS
WAOB
FAGEP
FCITWP
FFODP
FHISP
FINDP
FINTP
FJWDP
FJWMNP
FJWRIP
FLANP
FMARHYP
FMIGSP
FMILPP
FMILSP
FOCCP
FOIP
FPAP
FPERNP
FPINCP
FPOBP
FPOWSP
FRACP
FRELP
FRETP
FSEMP
FSSIP
FSSP
FWAGP
FWKHP
FYOEP


In [8]:
# reading in the household PUMS for referencing purposes
dfs = []
dfs.append(
    pd.read_csv(
        f"{directory}/{base_household_name}a_{year}.csv", usecols=filter_household_cols
    )
)
dfs.append(
    pd.read_csv(
        f"{directory}/{base_household_name}b_{year}.csv", usecols=filter_household_cols
    )
)

df_h = pd.concat(dfs).reset_index(drop=True)

del dfs

In [9]:
# cleaning missing identifier data
df_p["MIGPUMA"] = df_p["MIGPUMA"].fillna(0)
df_p["MIGSP"] = df_p["MIGSP"].fillna(0)

In [10]:
# filtering out the PUMS to people 18+ who moved from places in the contiguous united states to other places in the contiguous united states
mask = (
    (df_p["MIGSP"] <= 56)
    & (~df_p["MIGSP"].isin([2, 15]))  # 2 and 15 correspond to Alaska and Hawaii
    & (df_p["ST"] <= 56)
    & (~df_p["ST"].isin([2, 15]))
    # institutionalized GQ are outside of the context of this study
    & (~df_p["RELP"].isin([16]))
)
print(df_p.shape)
df_subset = df_p.loc[mask].copy()
print(df_subset.shape)

(3214539, 159)
(3101311, 159)


In [11]:
# origin is the MIGSP + MIGPUMA
# need ints since it is treated as a float by default
origin = df_subset["MIGSP"].astype(int).astype(str).str.zfill(2) + df_subset[
    "MIGPUMA"
].astype(int).astype(str).str.zfill(5)
# chosen is the current location, ST + PUMA
chosen = df_subset["ST"].astype(int).astype(str).str.zfill(2) + df_subset[
    "PUMA"
].astype(int).astype(str).str.zfill(5)
# people who stayed had their origin is all zeros (due to fillna 0)
df_subset["ORIGIN"] = origin  # migpuma geography
df_subset["CHOSEN"] = chosen  # puma geography

In [12]:
puma_migpuma = lio.load_puma_migpuma("../geometry/equivalencies/puma_migpuma_2010.csv")
puma_migpuma.head()

,State,MIGPUMA
PUMA,,
0100100,01,0100190
0100200,01,0100290
0100301,01,0100290
0100302,01,0100290
0100400,01,0100400


In [13]:
df_subset["ORIGIN"].value_counts()

ORIGIN
0000000    2726512
0603700       9975
0400100       5616
1703400       5487
0800190       5253
            ...   
2201400         74
0501900         72
2101000         71
4806400         70
5401300         65
Name: count, Length: 976, dtype: int64

In [14]:
# backfill the stay origins to the MIGPUMA where they are currently (chosen == origin)
df_subset["ORIGIN"] = np.where(
    df_subset["ORIGIN"] == "0000000",
    puma_migpuma.loc[df_subset["CHOSEN"], "MIGPUMA"],
    df_subset["ORIGIN"],
)
# fill in the origin state with this backfill in place
df_subset["ORIGIN_STATE"] = df_subset["ORIGIN"].str[:2]
df_subset["CHOSEN_MIGPUMA"] = df_subset["CHOSEN"].map(puma_migpuma["MIGPUMA"])

# define STAY as moving outside the MIGPUMA
df_subset["STAY"] = df_subset["CHOSEN_MIGPUMA"] == df_subset["ORIGIN"]

In [15]:
df_subset["ORIGIN"].value_counts()

ORIGIN
0603700    100439
2500390     48474
1703400     41009
0400100     40465
4804600     36240
            ...  
2202100       779
2000700       776
4806900       768
2300600       734
0800400       718
Name: count, Length: 975, dtype: int64

In [16]:
df_subset["ORIGIN_STATE"].value_counts()

ORIGIN_STATE
06    368936
48    258948
12    194137
36    193489
42    125629
17    124126
39    115886
37     98955
26     97365
13     97017
34     86822
51     82185
53     74082
25     67967
04     66972
18     65803
47     65794
29     60397
55     58368
24     58286
27     54648
08     54466
45     47905
01     46208
21     43871
22     42203
41     40869
40     36259
09     35516
19     31339
49     30771
05     29484
20     28752
32     28137
28     28084
31     19024
35     18724
54     17559
16     16234
33     13387
23     12839
44     10126
30     10057
10      8859
46      8758
38      7714
11      6419
50      6296
56      5639
Name: count, dtype: int64

In [17]:
df_subset["CHOSEN"].value_counts()

CHOSEN
0102500    4488
5310200    4031
5500700    3994
5500100    3908
1200500    3207
           ... 
2701503     570
5541001     562
2701403     558
2701402     511
4203207     493
Name: count, Length: 2336, dtype: int64

In [18]:
df_subset["STAY"].value_counts()

STAY
True     2947963
False     153348
Name: count, dtype: int64

In [19]:
# should not get duplicate measurements for the same SERIALNO
assert df_h["SERIALNO"].value_counts().max() == 1
# merge the household info into df
df = pd.merge(df_subset, df_h, left_on="SERIALNO", right_on="SERIALNO", how="left")

In [20]:
def agreement(sub, name, relps=None):
    """Share of households where all listed members agree, on ORIGIN and on STAY."""
    if relps is not None:
        sub = sub[sub["RELP"].isin(relps)]
        n = sub.groupby("SERIALNO")["RELP"].nunique()
        sub = sub[sub["SERIALNO"].isin(n[n == len(relps)].index)]
    size = sub.groupby("SERIALNO").size()
    sub = sub[sub["SERIALNO"].isin(size[size >= 2].index)]
    g = sub.groupby("SERIALNO")[["ORIGIN", "STAY"]].nunique()
    # this is on nunique
    a_o = (g["ORIGIN"] == 1).mean()
    a_s = (g["STAY"] == 1).mean()
    print(
        f"{name:34s} n={len(g):>8,}  origin agree={a_o:.4f}  stay agree={a_s:.4f}  gap={a_s - a_o:.4f}"
    )


# a_o (ORIGIN agreement): everybody reports the same origin MIGPUMA. Splits into
#   households that never left, and households that moved as a unit. The latter
#   is the population the joint-decision model is about.
#
# a_s (STAY agreement): everybody all-stayed or all-moved. Weaker than a_o, since
#   STAY collapses every distinct origin into one "moved" bucket. In practice
#   a_s is ~"everyone stayed": 95% of persons have STAY=True, and the all-moved
#   branch is almost entirely already inside a_o.
#
# gap = a_s - a_o = 0.0033: all-moved households arriving from DIFFERENT origins
#   (in-migrants converging to form a household). A lower bound on merging --
#   the commoner pattern is a joiner moving into a household that never left
#   (33,029 / 0.0367), which disagrees on STAY and so falls outside the gap.
#
# the households outside origin agree did things like have people move in externally with a person who stayed
agreement(df, "all multi-person households")
agreement(df, "householder + spouse", [0, 1])
agreement(df, "householder + unmarried partner", [0, 13])
agreement(df, "householder + own child", [0, 2])
agreement(df, "householder + roommate/boarder", [0, 12])
agreement(df, "householder + other relative", [0, 10])

# --- 3. adults only, for comparison with the modeled sample -------------------
print()
agreement(df[df["AGEP"] >= 18], "adults only: all multi-person")
agreement(df[df["AGEP"] >= 18], "adults only: hh + spouse", [0, 1])

all multi-person households        n= 900,789  origin agree=0.9601  stay agree=0.9633  gap=0.0033
householder + spouse               n= 633,446  origin agree=0.9946  stay agree=0.9952  gap=0.0006
householder + unmarried partner    n=  72,552  origin agree=0.9215  stay agree=0.9349  gap=0.0133
householder + own child            n= 429,169  origin agree=0.9751  stay agree=0.9759  gap=0.0008
householder + roommate/boarder     n=  29,423  origin agree=0.8049  stay agree=0.8357  gap=0.0308
householder + other relative       n=  23,727  origin agree=0.9054  stay agree=0.9100  gap=0.0046

adults only: all multi-person      n= 852,026  origin agree=0.9620  stay agree=0.9655  gap=0.0035
adults only: hh + spouse           n= 633,371  origin agree=0.9946  stay agree=0.9952  gap=0.0006


In [21]:
multi = df.groupby("SERIALNO").filter(lambda g: len(g) >= 2)
g = multi.groupby("SERIALNO").agg(
    n_origin=("ORIGIN", "nunique"),
    n_stay=("STAY", "nunique"),
    all_moved=("STAY", lambda s: (s == 0).all()),
)

coherent = g["n_origin"] == 1
merge_converge = (~coherent) & g[
    "all_moved"
]  # the "gap": all in-migrants, diff origins
merge_join = (~coherent) & (g["n_stay"] == 2)  # incumbent + joiner

for name, mask in [
    ("coherent unit", coherent),
    ("merger: all arrived, diff origins", merge_converge),
    ("merger: joined existing household", merge_join),
]:
    print(f"{name:38s} {mask.sum():>8,}  {mask.mean():.4f}")

coherent unit                           864,808  0.9601
merger: all arrived, diff origins         2,952  0.0033
merger: joined existing household        33,029  0.0367


In [22]:
df["RELP"].value_counts()

RELP
0     1245340
2      750898
1      633987
17      76036
13      72647
7       67887
12      40652
4       33605
10      33406
6       33001
15      31601
5       28992
3       19248
9       11576
11      11352
8        8707
14       2376
Name: count, dtype: int64

In [23]:
df["NP"].value_counts()

NP
2     916635
4     591672
3     556720
1     419620
5     333898
6     154107
7      65081
8      31066
9      14619
10      8124
11      4354
12      2759
13       998
14       558
15       374
20       260
17       238
16       174
18        54
Name: count, dtype: int64

In [24]:
adults = df[(df["RELP"] <= 15) & (df["AGEP"] >= 18)].copy()
adults = adults[adults.groupby("SERIALNO")["AGEP"].transform("size") > 1]

g = adults.groupby("SERIALNO")
ref = adults[adults["RELP"] == 0].set_index("SERIALNO")
w = ref["PWGTP"]


def report(label, mask):
    m = mask.reindex(ref.index).fillna(False)
    print(f"{label:<58} {m.mean():>7.2%} {w[m].sum() / w.sum():>8.2%}")


print(f"Multi-adult households: {len(ref):,}")
print(f"Mean adults per household: {g.size().reindex(ref.index).mean():.2f}\n")

# report(
#     "Any adult BA+ but reference person not",
#     (g["EDU_HAS_DEGREE"].max() == 1) & (ref["EDU_HAS_DEGREE"] == 0),
# )
# report("Household mixed on BA+", g["EDU_HAS_DEGREE"].nunique() > 1)
# report("More than one race group", g["RACE_ETHNICITY"].nunique() > 1)
# report("Age spread > 16 years", (g["AGEP"].max() - g["AGEP"].min()) > 16)

# # industry -- swap NAICS_GROUP for whatever your grouping column is called
# emp = adults[adults["NAICS_GROUP"].notna()]
# by_hh = emp.groupby("SERIALNO")["NAICS_GROUP"]
# report("More than one industry among employed", by_hh.nunique() > 1)
# report(
#     "Employed adult differs from reference person",
#     by_hh.apply(lambda s: (s != ref["NAICS_GROUP"].get(s.name)).any()),
# )

Multi-adult households: 851,698
Mean adults per household: 2.34



In [25]:
# Household movement concordance: for each household (SERIALNO), do all members
# share the same STAY status (all moved together vs. a mixed household)?
household = df.groupby("SERIALNO").agg(
    n_members=("STAY", "size"),
    n_unique_stay=("STAY", "nunique"),
    # no separate household weight column present -- approximated with each
    # household"s first member"s PWGTP; swap for WGTP if it ever gets merged in
    weight=("PWGTP", "first"),
)
household["unanimous"] = household["n_unique_stay"] == 1

n_households = len(household)
print(f"Total households: {n_households}")
print(
    f"Unanimous (everyone moved or everyone stayed): {household['unanimous'].mean():.2%}"
)
print(f"Mixed (some moved, some stayed): {(~household['unanimous']).mean():.2%}")

weighted_unanimous = (
    household.loc[household["unanimous"], "weight"].sum() / household["weight"].sum()
)
print(f"Weighted share of unanimous households: {weighted_unanimous:.2%}")

# single-person households are trivially unanimous, so also look at multi-person only
multi_person = household[household["n_members"] > 1]
print(
    f"\nAmong multi-person households: {multi_person['unanimous'].mean():.2%} unanimous"
)

# among unanimous households, how many moved together vs. stayed together
unanimous_ids = household[household["unanimous"]].index
unanimous_status = (
    df[df["SERIALNO"].isin(unanimous_ids)]
    .drop_duplicates("SERIALNO")
    .set_index("SERIALNO")["STAY"]
)
print("\nAmong unanimous households, share where everyone moved vs everyone stayed:")
print(unanimous_status.value_counts(normalize=True))

Total households: 1322764
Unanimous (everyone moved or everyone stayed): 97.50%
Mixed (some moved, some stayed): 2.50%
Weighted share of unanimous households: 97.41%

Among multi-person households: 96.33% unanimous

Among unanimous households, share where everyone moved vs everyone stayed:
STAY
True     0.948879
False    0.051121
Name: proportion, dtype: float64


In [26]:
df[~df.SERIALNO.map(household["unanimous"]) & (df["STAY"] == 0)]["RELP"].value_counts()

RELP
2     10524
0      9272
12     4079
1      3934
13     3339
7      2715
15     2549
10     2147
5      1371
6      1150
4      1139
11      965
9       857
8       459
3       320
14      268
Name: count, dtype: int64

In [27]:
df[~df.SERIALNO.map(household["unanimous"]) & (df["STAY"] != 0)]["RELP"].value_counts()

RELP
0     23710
2     15698
1     10080
12     4833
7      3193
13     2734
15     2262
10     1659
5      1191
6      1156
4       983
11      819
3       548
9       538
8       314
14      217
Name: count, dtype: int64

In [28]:
df["SFN"].value_counts()

SFN
1.0    94698
2.0     1674
3.0       42
4.0        4
Name: count, dtype: int64

In [29]:
"""Baseline (unit = whole household)

persons            : 2,386,856
households         : 1,246,447
decision units     : 1,283,126
reduction vs person: 46.2%
unit size dist     : {1: 428073, 2: 673385, 3: 131399, 4: 38299, 5: 9004, 6: 2030, 7: 553, 8: 198} -> this excludes children under 18
multi-person units : 855,053  share agreeing on ORIGIN: 0.9639

misattributed people (tie-free) : 33,460  (0.0140 of all)
flagged, all people                33,460  (0.0140 of 2,386,856)  weighted=4,024,287
flagged, in multi-person units     33,460  (0.0171 of 1,958,783)  weighted=4,024,287

misattributed movers in multi-person units: 27,009 (0.3196 of 84,510)  weighted=3,233,428

RELP of misattributed movers (2-person units excluded -- tie-break there
is arbitrary, so 'which' member is blamed carries no meaning):
RELP
2     4613
12    2987
15    1566
0     1270
10    1029
5      746
11     644
6      628
7      543
4      488
Name: count, dtype: int64

1,553,274 within-household pairs
people separated from a same-origin housemate: 121,008
  of which involve a mover: 1,375
"""

"Baseline (unit = whole household)\n\npersons            : 2,386,856\nhouseholds         : 1,246,447\ndecision units     : 1,283,126\nreduction vs person: 46.2%\nunit size dist     : {1: 428073, 2: 673385, 3: 131399, 4: 38299, 5: 9004, 6: 2030, 7: 553, 8: 198} -> this excludes children under 18\nmulti-person units : 855,053  share agreeing on ORIGIN: 0.9639\n\nmisattributed people (tie-free) : 33,460  (0.0140 of all)\nflagged, all people                33,460  (0.0140 of 2,386,856)  weighted=4,024,287\nflagged, in multi-person units     33,460  (0.0171 of 1,958,783)  weighted=4,024,287\n\nmisattributed movers in multi-person units: 27,009 (0.3196 of 84,510)  weighted=3,233,428\n\nRELP of misattributed movers (2-person units excluded -- tie-break there\nis arbitrary, so 'which' member is blamed carries no meaning):\nRELP\n2     4613\n12    2987\n15    1566\n0     1270\n10    1029\n5      746\n11     644\n6      628\n7      543\n4      488\nName: count, dtype: int64\n\n1,553,274 within-h

In [40]:
PRIMARY = {0, 1, 13}
CHILDREN = {2, 3, 4, 7, 14}  # children, foster chidlren, grandchildren
ALL_PRIMARY = PRIMARY | CHILDREN
df["UNIT"] = np.where(
    df.SFN.notna(),
    # subfamily → its own unit
    df.SERIALNO + "_SF" + df.SFN.fillna(-99999).astype(int).astype(str),
    np.where(
        df.RELP.isin(PRIMARY) | (df.RELP.isin(CHILDREN) & (df.AGEP < 25)),
        # primary family, non-subfamily
        df.SERIALNO + "_P",
        # treat everyone else as singletons
        df.SERIALNO + "_I" + df.SPORDER.astype(int).astype(str),
    ),
)

In [41]:
# working frame for the whole evaluation section (drops GQ since there truly aren't any aggreggate decision units there; keeps children)
d = df[(df["RELP"] < 16) & (df["AGEP"] >= 18)][
    ["SERIALNO", "SPORDER", "SFN", "RELP", "ORIGIN", "STAY", "UNIT", "PWGTP"]
].copy()
d["PID"] = d["SERIALNO"] + "_" + d["SPORDER"].astype(int).astype(str)

sizes = d.groupby("UNIT").size()
print(f"persons            : {len(d):,}")
print(f"households         : {d['SERIALNO'].nunique():,}")
print(f"decision units     : {d['UNIT'].nunique():,}")
print(f"reduction vs person: {100 * (1 - d['UNIT'].nunique() / len(d)):.1f}%")
print(f"unit size dist     : {sizes.value_counts().sort_index().head(8).to_dict()}")

u = d.groupby("UNIT")["ORIGIN"].nunique()
multi_u = sizes[sizes >= 2].index
print(
    f"multi-person units : {len(multi_u):,}  "
    f"share agreeing on ORIGIN: {(u.loc[multi_u] == 1).mean():.4f}"
)

# ----------------------------------------------------------------------------
# persons            people left after dropping GQ and under-18s. The estimation
#                    sample if you modelled individuals.
# households         distinct SERIALNOs. Not the decision unit -- shown only so the
#                    next line can be compared against it.
# decision units     rows you would actually estimate on. Always >= households,
#                    beacause the logic splits household units into 1 or more decision units
# reduction vs person how much the grouping shrinks the sample. Pure cost: it is the
#                    power you trade away for a more defensible decision unit.
#                    Compare against the baseline (~46% for household-as-unit); a
#                    SMALLER reduction means a stricter, more conservative rule.
# unit size dist     units of size 1 are people modelled individually anyway -- they
#                    cannot be misattributed and dilute every rate computed over
#                    "all people". Watch how many units are size 1 (excluding children)
# share agreeing     of multi-person units, the fraction where every member reports
#                    the same ORIGIN.


persons            : 2,386,856
households         : 1,246,447
decision units     : 1,541,357
reduction vs person: 35.4%
unit size dist     : {1: 791585, 2: 670596, 3: 64357, 4: 13253, 5: 1419, 6: 132, 7: 13, 9: 2}
multi-person units : 749,772  share agreeing on ORIGIN: 0.9822


### Reporting at the person level

Everything below counts **people**, not pairs.

There are exactly two ways the grouping rule can be wrong, and they are different
kinds of wrong. Both are measured against the same metric -- whether two people
report the same `ORIGIN` (the MIGPUMA they lived in a year ago), which indicates they are
(probably) a single deiciosn unit

---

#### misattributed -- the rule glued together people who do not belong together

A decision unit is a claim: *"these people faced one origin and jointly picked one
destination."* A member is **misattributed** when their own `ORIGIN` is not the
unit's origin -- the claim is false for them.

> A unit of three in Chicago. Two members lived in Phoenix a year ago; the third
> was already in Chicago and never moved. The unit's origin is Phoenix. That third
> person is misattributed.

When you estimate, that person contributes an observation reading *"faced origin
Phoenix, chose Chicago"* -- a choice situation they were never in. It is a
**fabricated observation**, and the MNL cannot tell it apart from a real one. This
is the dangerous error: it does not just add noise, it biases coefficients, because
the fake observations are not randomly distributed (they concentrate in households
that took in a mover).

Note the flag marks whoever is in the **minority** of their unit, not whoever did
the moving. In the example above the person flagged is the one who *stayed put*.
That is why the count includes both movers and stayers.

---

#### separated -- the rule split apart people who do belong together

Two people live together and report the same `ORIGIN`, but the rule assigned them to
different units.

> A householder and their 30-year-old sibling, both of whom moved from Denver to
> Atlanta together last year. `RELP=5` is not in `PRIMARY`, so the sibling becomes
> a singleton and the two are modelled separately.

The model now treats one joint decision as **two independent decisions** -- exactly
the assumption this whole exercise set out to escape. Nothing is fabricated; the
observations are real. You have simply failed to aggregate, so those people revert
to the individual-level model you started with.

---

#### Why they are never combined into one score

| | misattributed | separated |
|---|---|---|
| what happens | invents a choice nobody made | records a real choice, twice |
| statistical cost | **bias** -- wrong coefficients | **inefficiency** -- overstated sample, understated SEs |
| fallback if wrong | corrupted data | the old individual model |

Misattribution is strictly worse. A rule that removes 100 misattributions at the
cost of 100 separations is a good trade, so the two are always reported side by
side and never netted into a single F1-style number.

Every rule faces this tension: grouping more people catches more genuine joint
moves (fewer separated) but inevitably sweeps in people who do not belong (more
misattributed). Household-as-unit sits at one extreme; modelling everyone
individually sits at the other.

In [42]:
# MISATTRIBUTED = people the rule glued into a unit they do not belong to.
#
# A unit asserts "these people shared one origin and jointly chose one destination".
# A member is misattributed when their own ORIGIN is not the unit's origin, so that
# assertion is false for them. At estimation they contribute a choice situation they
# were never in -- a fabricated observation the MNL treats as real. This is the
# error that BIASES coefficients, which is why it is the number to minimise.
#
# The unit's origin = the origin the most members share (plurality). Anyone else is
# flagged. So the flag marks who is in the MINORITY, not who did the moving: if two
# people move in with someone who never left, the incumbent is the one flagged.

# How many people sit in a unit whose ORIGIN is not theirs?
#
# Headline count is tie-free: for each unit, (size - size of the largest single-ORIGIN
# bloc) is the number of members who cannot belong to the unit's plurality origin,
# regardless of which bloc you call "the" origin.
cnt = d.groupby(["UNIT", "ORIGIN"], sort=False).agg(
    n=("SPORDER", "size"), min_sp=("SPORDER", "min")
)
unit_tot = cnt.groupby("UNIT")["n"].sum()
unit_max = cnt.groupby("UNIT")["n"].max()
excess = (unit_tot - unit_max).sum()
print(f"misattributed people (tie-free) : {excess:,}  ({excess / len(d):.4f} of all)")

# Per-person flag, for the weighted and RELP breakdowns below. Tie-break is explicit:
# plurality first, then the earliest-listed member (SPORDER 1 = reference person).
ranked = cnt.reset_index().sort_values(
    ["UNIT", "n", "min_sp"], ascending=[True, False, True]
)
unit_origin = ranked.drop_duplicates("UNIT").set_index("UNIT")["ORIGIN"]
d["misattributed"] = d["ORIGIN"] != d["UNIT"].map(unit_origin)

grouped = d[d["UNIT"].map(sizes) > 1]  # singletons cannot be misattributed
for label, sub in [("all people", d), ("in multi-person units", grouped)]:
    n = int(sub["misattributed"].sum())
    w = sub.loc[sub["misattributed"], "PWGTP"].sum()
    print(
        f"flagged, {label:<22} {n:>9,}  ({n / len(sub):.4f} of {len(sub):,})"
        f"  weighted={w:,.0f}"
    )

# ----------------------------------------------------------------------------
# All three printed lines count THE SAME PEOPLE. They differ in method and
# denominator, not in what is being measured.
#
# "misattributed (tie-free)"  how MANY people are misattributed, without deciding
#                    WHICH ones. Per unit: size - (largest single-ORIGIN bloc). A
#                    2-person unit that disagrees contributes 1 without having to
#                    name the odd one out. Robust to tie-breaking.
#
# "flagged, all people"       the same count, but naming individuals (plurality
#                    first, then lowest SPORDER, so the reference person wins ties).
#                    Needed for the weighted and RELP breakdowns, which require
#                    pointing at specific rows. Identical to the line above by construction.
#
# "flagged, in multi-person"  same people, denominator restricted to units of 2+.
#                    Singletons cannot be misattributed, so including them only
#                    dilutes the rate. This is the honest denominator of the two.
#
# weighted=          PWGTP-weighted misattribution count, i.e. the national population estimate
#
# Note a stayer CAN be flagged: if two members moved in together from X and one
# person never left, the plurality origin is X and the incumbent is the odd one out.
# The flag marks who is in the minority, not who did the moving.

misattributed people (tie-free) : 13,529  (0.0057 of all)
flagged, all people                13,529  (0.0057 of 2,386,856)  weighted=1,436,137
flagged, in multi-person units     13,529  (0.0085 of 1,595,271)  weighted=1,436,137


In [43]:
# movers only (people)
mv = grouped[~grouped["STAY"].astype(bool)]
n = int(mv["misattributed"].sum())
print(
    f"misattributed movers in multi-person units: {n:,} "
    f"({n / len(mv):.4f} of {len(mv):,})  "
    f"weighted={mv.loc[mv['misattributed'], 'PWGTP'].sum():,.0f}"
)

print("\nRELP of misattributed movers (2-person units excluded -- tie-break there")
print("is arbitrary, so 'which' member is blamed carries no meaning):")
big = mv[mv["UNIT"].map(sizes) > 2]
print(big.loc[big["misattributed"], "RELP"].value_counts().head(10))

# ----------------------------------------------------------------------------
# "misattributed movers"  a subset of the total above, restricted to STAY == False.
#                    The remainder (total - this) are misattributed stayers: people
#                    outnumbered in their own unit by members who arrived from
#                    elsewhere.
#
# the rate           misattributed movers / movers in multi-person units. This is
#                    the fraction of the identifying sample carrying a fabricated
#                    destination choice -- a mover credited with a move they did not
#                    make. Compare it against the baseline rate (~0.32 for
#                    household-as-unit); the drop is what the rule buys you.
#
# RELP breakdown     which relationship types the residual error sits in. Units of
#                    3+ only, because in a 2-person unit the tie-break decides who
#                    gets blamed and that choice is arbitrary. Read it as "where is
#                    the rule still failing":
#                      2/3/4 (own children)  -- adults living with parents. Since
#                        under-18s are excluded these are grown children who moved
#                        in or out independently. Currently ~74% of the residual and
#                        untouched by any PRIMARY variant tested.
#                      0 (householder)       -- outnumbered in their own unit; the
#                        household re-formed around members from another origin.
#                      12/15/10/5            -- housemates, nonrelatives, other
#                        relatives, siblings. If these are large the rule is not
#                        splitting off the people it was designed to split off.


misattributed movers in multi-person units: 10,003 (0.1684 of 59,405)  weighted=1,060,789

RELP of misattributed movers (2-person units excluded -- tie-break there
is arbitrary, so 'which' member is blamed carries no meaning):
RELP
2     2144
7      298
4      242
0      133
1      115
13     103
3       55
14       8
Name: count, dtype: int64


In [44]:
# SEPARATED = people the rule split apart who do belong together.
#
# Two people live in the same household and report the same ORIGIN -- they plausibly
# moved as one -- but the rule put them in different units. Nothing is fabricated;
# both observations are real. The model just treats one joint decision as two
# independent ones, which is the assumption this exercise set out to escape. Those
# people revert to the individual-level model.
#
# Cost is INEFFICIENCY, not bias: the sample looks larger than it is, so standard
# errors come out too small. Strictly less damaging than misattribution, which is
# why the two are reported separately and never averaged together.

# People split away from a housemate they actually share an ORIGIN with.
# This one is inherently relational, so it needs the within-household pair table to determine the number of offending pairs.
pairs = d.merge(d, on="SERIALNO", suffixes=("_a", "_b"))
pairs = pairs[pairs["SPORDER_a"] < pairs["SPORDER_b"]]
pairs["same_origin"] = pairs["ORIGIN_a"] == pairs["ORIGIN_b"]
pairs["same_unit"] = pairs["UNIT_a"] == pairs["UNIT_b"]
print(f"{len(pairs):,} within-household pairs")

split_pairs = pairs[~pairs["same_unit"] & pairs["same_origin"]]
separated = pd.unique(
    pd.concat([split_pairs["PID_a"], split_pairs["PID_b"]], ignore_index=True)
)
print(f"people separated from a same-origin housemate: {len(separated):,}")

mover_split = split_pairs[
    ~split_pairs["STAY_a"].astype(bool) | ~split_pairs["STAY_b"].astype(bool)
]
sep_mv = pd.unique(
    pd.concat([mover_split["PID_a"], mover_split["PID_b"]], ignore_index=True)
)
print(f"  of which involve a mover: {len(sep_mv):,}")

# ----------------------------------------------------------------------------
# This is the COST side. Misattribution puts fabricated choices into the data
# (bias); separation only forgoes aggregation, treating potentially a single decision unit as 2+
# decision units. This is arguably the lesser evil compared to misattribution.
#
# "within-household pairs"   size of the pair table. Diagnostic only.
#
# "people separated"         Pretty much just quantifies how much the rule separates people.
#                            Overwhelmingly dominated by stayers who have the same origin as they didn't move.
#
# "of which involve a mover" THIS is the real cost.
#
# Counted per PERSON, not per pair: one misplaced person in a 4-person unit would
# otherwise generate three bad pairs and triple-count the problem.


1,553,274 within-household pairs
people separated from a same-origin housemate: 613,368
  of which involve a mover: 9,086


In [ ]:
two = mv[mv["UNIT"].map(sizes) == 2]
pair_relp = (
    d[d["UNIT"].isin(two["UNIT"])]
    .groupby("UNIT")["RELP"]
    .agg(lambda s: tuple(sorted(s)))
)
print(
    pair_relp[pair_relp.index.isin(two.loc[two["misattributed"], "UNIT"])]
    .value_counts()
    .head(10)
)

RELP
(0, 13)    3846
(0, 1)     2017
(0, 2)      539
(0, 7)      215
(2, 9)      177
(6, 6)       24
(0, 3)       16
(0, 4)       13
(8, 8)       12
(4, 9)       11
Name: count, dtype: int64
